# SentinelAI — Notebook 07: LLM-based MedDRA Coding

**Goal:** Use a local LLM (llama3.2 via Ollama) as the final decision stage in the coding pipeline to select a single MedDRA Preferred Term (PT) from the reranked candidates and produce an interpretable confidence score.

**Complete pipeline (all stages):**
```
MAUDE narrative (free text)
        |
        v
[Stage 1] HybridSearcher       BM25 (pg_trgm) + Vector (pgvector/PubMedBERT)
          RRF fusion        --> Top-20 candidates
        |
        v
[Stage 2] CrossEncoder         MiniLM-L-6 scores each (query, PT) pair
          Reranker          --> Top-5 candidates
        |
        v
[Stage 3] LLM Coder            llama3.2 reads narrative + candidates
          (this notebook)   --> 1 PT code + confidence + rationale (JSON)
        |
        v
processed.coding_results (PostgreSQL)
```

---
## 1. Key Concepts

### Why an LLM as the final stage?

The CrossEncoder gives us a ranked list based on learned relevance — but it outputs a single **logit** with no explanation and no clinical reasoning. For a safety-critical application like adverse event coding, we need:

| Requirement | CrossEncoder | LLM |
|-------------|-------------|-----|
| Ranked candidates | Yes | Not needed (already done) |
| Reads full narrative | No (PT name only) | Yes |
| Understands negation | Limited | Yes |
| Structured output | No | Yes (with prompting) |
| Confidence score | Proxy only (score gap) | Explicit |
| Rationale / audit trail | No | Yes |
| Flags uncertain cases | No | Yes |

### Ollama

**Ollama** is a local LLM inference server — it runs open-source models (llama3.2, mistral, etc.) directly on your hardware with no API calls, no data leaving the server, and no per-token cost.

For SentinelAI this is critical: **MAUDE data contains PHI (Protected Health Information)** and should not be sent to external APIs. Ollama on the Hetzner server ensures Privacy-by-Design.

Ollama exposes a simple REST API at `http://localhost:11434`. We use the `/api/chat` endpoint with a system prompt + user message.

### llama3.2 — the model

**Llama 3.2** is Meta's open-source language model. The 3B parameter version (3.2:3b, ~2GB) is installed on our Hetzner server. It is small enough to run on CPU but capable enough for structured text tasks like selecting from a short candidate list.

**Parameters** = the learned weights of a neural network. Larger = more capable but slower and more RAM.  
3B parameters ≈ 2GB RAM (in 4-bit quantization). Sufficient for our task.

### Structured output via prompting

LLMs output free text by default. We force structured JSON output by:
1. **System prompt** — defines the task, rules, and exact JSON schema
2. **Low temperature (0.1)** — reduces randomness, makes output more deterministic
3. **Response parsing** — strips markdown code fences, validates required fields
4. **Fallback** — if JSON parsing fails, use the top CrossEncoder candidate with low confidence

### Temperature

**Temperature** controls randomness in LLM output sampling:
- Temperature 0.0 = always pick the most likely next token (deterministic)
- Temperature 1.0 = sample from the full distribution (creative but unpredictable)
- Temperature 0.1 = near-deterministic, small random variation

For coding tasks we want low temperature — reproducible, structured output.

### Confidence score and flagging

The LLM is asked to output a confidence score from 0.0 to 1.0. Cases below `confidence_threshold = 0.5` are flagged for human review. This creates a **human-in-the-loop** safety net for ambiguous narratives — important for a regulatory-adjacent application.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dotenv import load_dotenv
import psycopg2

load_dotenv()

def get_db_url():
    url = os.getenv('DATABASE_URL')
    if url:
        return url
    host = os.getenv('POSTGRES_HOST', 'localhost')
    port = os.getenv('POSTGRES_PORT', '5432')
    db   = os.getenv('POSTGRES_DB', 'vigilex')
    user = os.getenv('POSTGRES_USER', 'vigilex')
    pw   = os.getenv('POSTGRES_PASSWORD', '')
    return f'postgresql://{user}:{pw}@{host}:{port}/{db}'

conn = psycopg2.connect(get_db_url())
print('DB connected.')

---
## 2. Load All Three Pipeline Stages

In [ ]:
from src.vigilex.coding.hybrid_search import HybridSearcher, EmbeddingModel
from src.vigilex.coding.reranker import CrossEncoderReranker
from src.vigilex.coding.llm_coder import LLMCoder

# Stage 1: Hybrid search
print('Loading PubMedBERT...')
embed_model = EmbeddingModel()
searcher = HybridSearcher(conn, embedding_model=embed_model, candidate_pool=100)

# Stage 2: CrossEncoder reranker
print('Loading CrossEncoder...')
reranker = CrossEncoderReranker()

# Stage 3: LLM coder
# Ollama runs on the Hetzner server -- use SSH tunnel locally:
#   ssh -L 11434:localhost:11434 cap@46.225.109.99
# Or set OLLAMA_URL env var
OLLAMA_URL = os.getenv('OLLAMA_URL', 'http://localhost:11434')
print(f'Connecting to Ollama at {OLLAMA_URL}...')
coder = LLMCoder(ollama_url=OLLAMA_URL, model='llama3.2', confidence_threshold=0.5)

print('\nAll pipeline stages loaded.')

---
## 3. Inspect the System Prompt

The system prompt is the core of the LLM coding behaviour. Let's look at it explicitly.

In [ ]:
from src.vigilex.coding.llm_coder import SYSTEM_PROMPT, _build_user_prompt

print('=== SYSTEM PROMPT ===')
print(SYSTEM_PROMPT)

In [ ]:
# Preview a user prompt before sending to LLM
EXAMPLE_NARRATIVE = "Patient reported blood sugar dropped very low after insulin pump delivered unexpected bolus. Patient became confused and shaky."

# Get candidates for preview
candidates = searcher.search(EXAMPLE_NARRATIVE, top_k=20)
reranked   = reranker.rerank(EXAMPLE_NARRATIVE, candidates, top_k=5)

print('=== USER PROMPT (sent to LLM) ===')
print(_build_user_prompt(EXAMPLE_NARRATIVE, reranked))

---
## 4. Run the Full Pipeline on Test Cases

In [ ]:
TEST_CASES = [
    {
        'label': 'Clear clinical term',
        'narrative': 'Patient experienced hypoglycaemia after insulin bolus delivery. Blood glucose dropped to 2.1 mmol/L.',
    },
    {
        'label': 'Consumer language',
        'narrative': 'Blood sugar dropped very low, patient was shaking and confused. Had to eat sugar to recover.',
    },
    {
        'label': 'Device malfunction',
        'narrative': 'Insulin pump stopped delivering insulin without warning. Occlusion alarm did not trigger. Patient found in DKA.',
    },
    {
        'label': 'Skin reaction',
        'narrative': 'Significant redness, swelling and itching at the infusion site. Possible allergic reaction to adhesive.',
    },
    {
        'label': 'Ambiguous / multi-symptom',
        'narrative': 'Patient reported feeling unwell, nausea and dizziness. Later lost consciousness briefly. Device found operating normally.',
    },
    {
        'label': 'Negation test',
        'narrative': 'Patient did NOT experience hypoglycaemia. Device alarmed but glucose was within normal range. No adverse event.',
    },
]

results = []

for tc in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"[{tc['label']}]")
    print(f"Narrative: {tc['narrative'][:80]}...")

    # Stage 1
    candidates = searcher.search(tc['narrative'], top_k=20)
    # Stage 2
    reranked   = reranker.rerank(tc['narrative'], candidates, top_k=5)
    # Stage 3
    coded      = coder.code(tc['narrative'], reranked)

    flag = '*** FLAGGED FOR REVIEW ***' if coded.flagged else ''
    print(f"Result:    PT {coded.pt_code} — {coded.pt_name}")
    print(f"SOC:       {coded.soc_name}")
    print(f"Confidence:{coded.confidence:.2f}  {flag}")
    print(f"Rationale: {coded.rationale}")

    results.append({
        'label':      tc['label'],
        'narrative':  tc['narrative'],
        'pt_code':    coded.pt_code,
        'pt_name':    coded.pt_name,
        'soc_name':   coded.soc_name,
        'confidence': coded.confidence,
        'flagged':    coded.flagged,
        'rationale':  coded.rationale,
    })

df_results = pd.DataFrame(results)
print(f"\n\nDone. {len(df_results)} cases coded, {df_results['flagged'].sum()} flagged for review.")

---
## 5. Results Summary

In [ ]:
pd.set_option('display.max_colwidth', 60)
df_results[['label', 'pt_name', 'soc_name', 'confidence', 'flagged']].style \
    .background_gradient(subset=['confidence'], cmap='RdYlGn') \
    .applymap(lambda v: 'color: red; font-weight: bold' if v else '', subset=['flagged'])

In [ ]:
# Confidence scores visualised
fig, ax = plt.subplots(figsize=(10, 4))

colors = ['coral' if f else 'steelblue' for f in df_results['flagged']]
bars = ax.barh(df_results['label'], df_results['confidence'], color=colors, alpha=0.85)
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1, label='Flagging threshold (0.5)')
ax.set_xlabel('LLM Confidence Score')
ax.set_xlim(0, 1.1)
ax.set_title('MedDRA Coding Confidence by Test Case', fontsize=12)

flagged_patch = mpatches.Patch(color='coral',     label='Flagged for review')
ok_patch      = mpatches.Patch(color='steelblue', label='Auto-coded')
ax.legend(handles=[ok_patch, flagged_patch, plt.Line2D([0],[0],color='black',linestyle='--')],
          labels=['Auto-coded', 'Flagged for review', 'Threshold (0.5)'])

plt.tight_layout()
plt.show()

---
## 6. Negation Handling

Negation is one of the hardest problems in clinical NLP — *"patient did NOT experience hypoglycaemia"* should NOT be coded as Hypoglycaemia.  
The LLM is the only stage in our pipeline that can reliably handle this, because it reads the full narrative with context.

In [ ]:
negation_case = df_results[df_results['label'] == 'Negation test'].iloc[0]

print('Negation test result:')
print(f"  PT coded:   {negation_case['pt_name']}")
print(f"  Confidence: {negation_case['confidence']:.2f}")
print(f"  Flagged:    {negation_case['flagged']}")
print(f"  Rationale:  {negation_case['rationale']}")
print()
print('Expected: NOT Hypoglycaemia. LLM should either flag or code a device/monitoring PT.')
print('If it coded Hypoglycaemia with high confidence: prompt engineering needed.')

---
## 7. Raw LLM Output Inspection

Always check what the LLM actually said before trusting the parsed result.

In [ ]:
# Re-run one case and show raw LLM output
narrative = TEST_CASES[0]['narrative']
candidates = searcher.search(narrative, top_k=20)
reranked   = reranker.rerank(narrative, candidates, top_k=5)
coded      = coder.code(narrative, reranked)

print('=== RAW LLM RESPONSE ===')
print(coded.raw_response)
print()
print('=== PARSED RESULT ===')
print(f'PT:         {coded.pt_code} — {coded.pt_name}')
print(f'SOC:        {coded.soc_name}')
print(f'Confidence: {coded.confidence}')
print(f'Rationale:  {coded.rationale}')
print(f'Flagged:    {coded.flagged}')

---
## 8. End-to-End Latency

How long does the full pipeline take per MAUDE report? This determines coding worker throughput.

In [ ]:
import time

BENCHMARK_NARRATIVE = TEST_CASES[0]['narrative']
N_RUNS = 3  # LLM is slow -- keep low

stage_times = {'hybrid': [], 'reranker': [], 'llm': [], 'total': []}

for _ in range(N_RUNS):
    t0 = time.time()

    t1 = time.time()
    cands = searcher.search(BENCHMARK_NARRATIVE, top_k=20)
    stage_times['hybrid'].append(time.time() - t1)

    t2 = time.time()
    reranked = reranker.rerank(BENCHMARK_NARRATIVE, cands, top_k=5)
    stage_times['reranker'].append(time.time() - t2)

    t3 = time.time()
    coder.code(BENCHMARK_NARRATIVE, reranked)
    stage_times['llm'].append(time.time() - t3)

    stage_times['total'].append(time.time() - t0)

print(f'End-to-end latency ({N_RUNS} runs):')
print(f"  {'Stage':<12} {'Mean (ms)':>12}")
print(f"  {'-'*26}")
for stage, times in stage_times.items():
    mean_ms = sum(times) / len(times) * 1000
    print(f"  {stage:<12} {mean_ms:>12.0f}")

---
## 9. Write Results to processed.coding_results

The final step: persist coding results to the DB for use in signal detection (Module 3).

In [ ]:
# Example: write one result to DB (in production this runs in the coding worker)
# Replace mdr_report_key with a real key from raw.maude_reports

EXAMPLE_MDR_KEY = 'TEST-NB07-001'   # placeholder -- use real key in production

INSERT_SQL = """
    INSERT INTO processed.coding_results
        (mdr_report_key, pt_code, pt_name, soc_name, llm_confidence, final_confidence, model_version)
    VALUES
        (%(key)s, %(pt_code)s, %(pt_name)s, %(soc_name)s, %(conf)s, %(conf)s, %(model)s)
    ON CONFLICT DO NOTHING
"""

# Re-run on example narrative
cands   = searcher.search(EXAMPLE_NARRATIVE, top_k=20)
reranked = reranker.rerank(EXAMPLE_NARRATIVE, cands, top_k=5)
coded    = coder.code(EXAMPLE_NARRATIVE, reranked)

with conn.cursor() as cur:
    cur.execute(INSERT_SQL, {
        'key':    EXAMPLE_MDR_KEY,
        'pt_code': coded.pt_code,
        'pt_name': coded.pt_name,
        'soc_name': coded.soc_name,
        'conf':    coded.confidence,
        'model':   'llama3.2:3b',
    })
conn.commit()

print(f'Written to processed.coding_results:')
print(f'  mdr_report_key: {EXAMPLE_MDR_KEY}')
print(f'  pt_code:        {coded.pt_code}')
print(f'  pt_name:        {coded.pt_name}')
print(f'  confidence:     {coded.confidence}')

---
## 10. Summary

**Module 2 (MedDRA Coding Engine) is now complete:**

| Stage | File | Status |
|-------|------|--------|
| MedDRA import | `import_meddra.py` | Done |
| PubMedBERT embeddings | `embed_meddra_terms.py` | Done |
| Hybrid search (BM25 + Vector + RRF) | `hybrid_search.py` | Done |
| CrossEncoder reranker | `reranker.py` | Done |
| LLM final coding | `llm_coder.py` | Done |
| Coding worker (production) | `workers/coding.py` | **Next** |

**Next step — Module 3 (Signal Detection):**
- PRR / ROR disproportionality analysis on `processed.coding_results`
- Grafana dashboard for signal visualisation
- Streamlit frontend with RAG natural language interface